# Tuần 1 — Bài toán, dữ liệu và thước đo
**Đồ án cá nhân Máy học: dự đoán khách hàng đồng ý gửi tiết kiệm có kỳ hạn** · thời lượng dự kiến: **3 giờ** (2 giờ notebook, 1 giờ viết nháp Mục 1–2 của báo cáo)

**Mục tiêu.** Sau tuần này, sinh viên có thể:
1. trình bày bài toán và vai trò của ba tập train / validation / test;
2. mô tả dữ liệu dạng bảng bằng số liệu (khám phá dữ liệu, *exploratory data analysis* — EDA);
3. nhận ra một thuộc tính gây **rò rỉ dữ liệu** (*data leakage*);
4. tính Precision, Recall, F1 của lớp `yes` cho một mô hình mốc và giải thích vì sao Accuracy gây hiểu lầm.

**Cách dùng notebook.** Mỗi mục được đánh dấu bằng một nhãn:
`[KIẾN THỨC]` nội dung cần đọc · `[MINH HỌA]` ô mã có sẵn, chạy và quan sát kết quả ·
`[YÊU CẦU]` phần sinh viên thực hiện: thay mỗi chỗ `None` hoặc `____` bằng mã phù hợp ·
`[TỰ KIỂM TRA]` ô kiểm tra tự động, in **ĐẠT** khi kết quả đúng ·
`[BÁO CÁO]` ghi câu trả lời ngắn vào ô văn bản để đưa vào báo cáo ·
`[CÂU HỎI SUY NGẪM]` câu hỏi để sinh viên tự trả lời trước; gợi ý ở cuối notebook.

> **Sổ tay thư viện** (`00_so_tay_thu_vien.pdf`, phát kèm đồ án) — các mục cần tra cứu trong tuần này: mục 1 (Colab); mục 2 (đọc, mô tả dữ liệu); mục 3 (vẽ hình); mục 4 (chia dữ liệu, tránh rò rỉ); mục 7 (đánh giá mô hình); mục 10 (đọc thông báo lỗi). Mỗi mục có đoạn mã mẫu chạy trên chính dữ liệu của đồ án, kết quả in ra và các lỗi thường gặp.
>
> Sinh viên chưa quen Colab, pandas hoặc scikit-learn được khuyến nghị làm trước notebook `tuan0_lam_quen_sinhvien.ipynb` (1–1,5 giờ, không chấm điểm, không nộp).

## 1.1 `[KIẾN THỨC]` Bài toán
Một ngân hàng gọi điện mời khách hàng mở **tài khoản tiết kiệm có kỳ hạn**. Phần lớn cuộc gọi không thành công, nên ngân hàng muốn một mô hình **dự đoán trước khi gọi** khách nào có khả năng đồng ý (`yes`) để ưu tiên gọi những khách đó.

Mô hình có thể mắc hai loại lỗi:
- **bỏ lỡ**: khách sẽ đồng ý nhưng mô hình dự đoán `no`, ngân hàng mất một khách hàng tiềm năng;
- **gọi nhầm**: khách sẽ từ chối nhưng mô hình dự đoán `yes`, nhân viên tốn một cuộc gọi không thành công.

Thước đo chính của đồ án là **F1 của lớp yes** (F1-yes), cân bằng giữa hai loại lỗi.

## Thiết lập
`[MINH HỌA]` Ô dưới đây nạp thư viện, tải dữ liệu và chia thành ba tập. Mã này **do giảng viên cung cấp, không thuộc phần đánh giá**; sinh viên chạy ô này mỗi khi mở notebook (khoảng 15 giây ở lần đầu).

In [ ]:
import io, json, zipfile, hashlib, datetime, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)

# Dữ liệu UCI Bank Marketing, tệp bank-full.csv; mã SHA-256 bảo đảm mọi sinh viên dùng cùng một tệp
NGUON = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
SHA256 = "d1513ec63b385506f7cfce9f2c5caa9fe99e7ba4e8c3fa264b3aaf0f849ed32d"

def doc_du_lieu():
    """Đọc du_lieu/bank-full.csv; nếu chưa có thì tải từ UCI (tệp nén lồng hai lớp) và lưu lại."""
    tep = Path("du_lieu/bank-full.csv")
    if not tep.exists():
        tep.parent.mkdir(exist_ok=True)
        ngoai = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(NGUON, timeout=30).read()))
        trong = zipfile.ZipFile(io.BytesIO(ngoai.read("bank.zip")))
        tep.write_bytes(trong.read("bank-full.csv"))
    assert hashlib.sha256(tep.read_bytes()).hexdigest() == SHA256, "Tệp dữ liệu không đúng phiên bản của đồ án."
    df = pd.read_csv(tep, sep=";")
    return df.assign(ma_dong=[f"dong:{i + 1:05d}" for i in range(len(df))])

du_lieu = doc_du_lieu()
train_val, test = train_test_split(du_lieu, test_size=0.2, stratify=du_lieu["y"], random_state=42)
train, val = train_test_split(train_val, test_size=0.25, stratify=train_val["y"], random_state=42)
train, val, test = (d.reset_index(drop=True) for d in (train, val, test))
print("Kích thước train / validation / test:", len(train), len(val), len(test), "(theo đề: 27126 / 9042 / 9043)")

# Thuộc tính đầu vào của mô hình (đã loại duration — lý do ở tuần 1, mục 1.6)
THUOC_TINH_SO = ["age", "balance", "day", "campaign", "pdays", "previous"]
THUOC_TINH_PL = ["job", "marital", "education", "default", "housing", "loan", "contact", "month", "poutcome"]

def chi_so(y_that, y_du_doan):
    """Các chỉ số chính, với yes là lớp cần phát hiện (tính bằng scikit-learn)."""
    p, r, f, _ = precision_recall_fscore_support(y_that, y_du_doan, labels=["yes"], zero_division=0)
    return {"accuracy": accuracy_score(y_that, y_du_doan), "precision_yes": p[0], "recall_yes": r[0], "f1_yes": f[0]}

`[YÊU CẦU]` Sinh viên điền mã số sinh viên. Ba chữ số cuối là **seed cá nhân**, dùng khi lấy mẫu ngẫu nhiên.

In [ ]:
MSSV = "____"          # YÊU CẦU: điền mã số sinh viên
S_SV = int(MSSV[-3:]) if MSSV[-3:].isdigit() else None
assert S_SV is not None, "CHƯA ĐẠT: cần điền MSSV rồi chạy lại ô này."
print("ĐẠT: seed cá nhân S_SV =", S_SV)

## 1.2 `[KIẾN THỨC]` Ba tập dữ liệu và các thuộc tính
| Tập | Vai trò | Quy tắc |
|---|---|---|
| **train** | mô hình **học** | mọi bước học từ dữ liệu (kể cả tiền xử lý) chỉ dùng train |
| **validation** | **so sánh và lựa chọn** mô hình, siêu tham số, ngưỡng | được dùng nhiều lần |
| **test** | **đánh giá cuối**, đóng vai khách hàng mới | chỉ dùng **một lần** ở tuần 4 |

Dữ liệu được chia ngẫu nhiên theo tỉ lệ 60 / 20 / 20, **phân tầng** (*stratified*) theo nhãn để ba tập có cùng tỉ lệ `yes`.

`[MINH HỌA]` Mô tả các thuộc tính (dùng cho **Bảng 2** của báo cáo):

In [ ]:
MO_TA = {"age": "tuổi", "job": "nghề nghiệp", "marital": "tình trạng hôn nhân (divorced gồm cả góa)",
         "education": "trình độ học vấn", "default": "có nợ xấu không", "balance": "số dư bình quân năm (euro)",
         "housing": "có vay mua nhà không", "loan": "có vay cá nhân không", "contact": "kênh liên lạc",
         "day": "ngày trong tháng của lần liên lạc cuối", "month": "tháng của lần liên lạc cuối",
         "duration": "thời lượng cuộc gọi cuối (giây)", "campaign": "số lần liên lạc trong chiến dịch này",
         "pdays": "số ngày từ lần liên lạc ở chiến dịch trước (-1: chưa từng)", "previous": "số lần liên lạc trước chiến dịch này",
         "poutcome": "kết quả chiến dịch trước", "y": "NHÃN: khách có gửi tiết kiệm có kỳ hạn không"}
pd.DataFrame({"mô tả": MO_TA, "kiểu": du_lieu.dtypes.astype(str), "số giá trị khác nhau": du_lieu.nunique()}).loc[list(MO_TA)]

## 1.3 `[YÊU CẦU]` Tỉ lệ khách đồng ý trong mỗi tập
Yêu cầu sinh viên thực hiện: tính tỉ lệ khách có nhãn `yes` của tập validation và tập test, theo mẫu đã có cho tập train (dùng cho **Bảng 1** và **Hình 1**).

In [ ]:
ty_le_yes = {
    "train": (train["y"] == "yes").mean(),
    "validation": None,     # YÊU CẦU: tỉ lệ yes của tập validation
    "test": None,           # YÊU CẦU: tỉ lệ yes của tập test
}
pd.Series(ty_le_yes)

In [ ]:
# [TỰ KIỂM TRA]
expected_ty_le_yes = {"train": (train["y"] == "yes").mean(), "validation": (val["y"] == "yes").mean(), "test": (test["y"] == "yes").mean()}
assert all(isinstance(v, (float, np.floating)) and np.isclose(ty_le_yes[k], expected_ty_le_yes[k], rtol=0, atol=1e-12) for k, v in ty_le_yes.items()), "CHƯA ĐẠT: cần đúng tỉ lệ yes của từng tập."
print("ĐẠT: đã có tỉ lệ yes chính xác của ba tập.")
dem = pd.DataFrame({ten: d["y"].value_counts() for ten, d in [("train", train), ("validation", val), ("test", test)]}).T
ax = dem[["no", "yes"]].plot.bar(rot=0, figsize=(6, 3.5), color=["#9aa5b1", "#2a78d6"], title="Phân bố nhãn trong ba tập (Hình 1)")
for c in ax.containers:
    ax.bar_label(c, fontsize=8)
ax.set_ylabel("số khách"); plt.tight_layout(); plt.show()

`[BÁO CÁO]` Ghi lại ba tỉ lệ. Vì sao ba tỉ lệ gần như bằng nhau? Một tập dữ liệu có khoảng 12% nhãn `yes` gây khó khăn gì cho việc đánh giá mô hình?

*Trả lời của sinh viên:* …

## 1.4 `[MINH HỌA]` Giá trị "unknown" và ví dụ trong dữ liệu
Dữ liệu không có ô trống, nhưng một số thuộc tính phân loại dùng giá trị `unknown` cho thông tin không có. Đồ án giữ `unknown` như **một nhóm riêng**, vì việc thiếu thông tin cũng có thể mang ý nghĩa (dùng cho **Bảng 3**).

In [ ]:
(train[THUOC_TINH_PL] == "unknown").mean().round(3).sort_values(ascending=False).to_frame("tỉ lệ unknown (train)")

Mỗi nhãn lấy 5 khách ngẫu nhiên bằng seed cá nhân, nên mỗi sinh viên quan sát một bộ ví dụ khác nhau (dùng cho **Bảng 4**).

In [ ]:
train.groupby("y").sample(5, random_state=S_SV)[["ma_dong", "age", "job", "balance", "contact", "month", "campaign", "poutcome", "y"]]

## 1.5 `[YÊU CẦU]` Tỉ lệ đồng ý theo nhóm khách hàng
Cột `dong_y` (1 nếu `yes`, 0 nếu `no`) đã được tạo sẵn; trung bình của cột này trong một nhóm chính là **tỉ lệ đồng ý** của nhóm đó. Ví dụ cho thuộc tính `contact` đã có sẵn.
Yêu cầu sinh viên thực hiện: tính tỉ lệ đồng ý theo `poutcome` (kết quả chiến dịch trước) và theo `month` (dùng cho **Bảng 5**).

In [ ]:
train["dong_y"] = (train["y"] == "yes").astype(int)
train.groupby("contact")["dong_y"].agg(["mean", "size"]).round(3)

In [ ]:
ty_le_poutcome = None    # YÊU CẦU: tỉ lệ đồng ý theo poutcome (theo mẫu ở ô trên)
ty_le_month = None       # YÊU CẦU: tỉ lệ đồng ý theo month
ty_le_poutcome

In [ ]:
# [TỰ KIỂM TRA]
assert ty_le_poutcome is not None and set(ty_le_poutcome.index) == set(train["poutcome"]), "CHƯA ĐẠT: ty_le_poutcome phải có một dòng cho mỗi giá trị của poutcome."
assert ty_le_month is not None and len(ty_le_month) == 12, "CHƯA ĐẠT: ty_le_month phải có 12 dòng."
print("ĐẠT: đã có tỉ lệ đồng ý theo poutcome và month.")
ty_le_month.sort_values().round(3).to_frame("tỉ lệ đồng ý").T

`[MINH HỌA]` Tỉ lệ đồng ý theo **nhóm tuổi**, dùng làm **Hình 2** của báo cáo. Hình này được dùng lại khi viết giả thuyết ở tuần 3.

In [ ]:
NHOM_TUOI = [0, 25, 30, 40, 50, 60, 200]
theo_tuoi = train.groupby(pd.cut(train["age"], NHOM_TUOI), observed=True)["dong_y"].agg(["mean", "size"])
ax = theo_tuoi["mean"].plot.bar(rot=0, figsize=(6, 3.5), color="#2a78d6", label="nhóm tuổi", title="Tỉ lệ đồng ý theo nhóm tuổi (train)")
ax.axhline(train["dong_y"].mean(), color="gray", linestyle="--", label="toàn bộ train"); ax.legend()
ax.set_xlabel("nhóm tuổi"); ax.set_ylabel("tỉ lệ đồng ý"); plt.tight_layout(); plt.show()
theo_tuoi.round(3).T

`[BÁO CÁO]` Viết **ba nhận xét** về dữ liệu train. Mỗi nhận xét cần **một con số** làm dẫn chứng; **ít nhất một nhận xét về tuổi** (Hình 2). Ví dụ về một nhận xét đạt yêu cầu: *"Khách có kết quả chiến dịch trước là success đồng ý với tỉ lệ x%, gấp khoảng k lần mức chung y%."*

1. …
2. …
3. …

`[CÂU HỎI SUY NGẪM]` (Câu 1) Tháng 3 có tỉ lệ đồng ý rất cao, tháng 5 rất thấp. Có nên kết luận "nên gọi khách vào tháng 3" không? Gợi ý: xem cột `size` (số khách) của từng tháng.

## 1.6 `[KIẾN THỨC]` Rò rỉ dữ liệu qua một thuộc tính
**Rò rỉ dữ liệu** xảy ra khi mô hình được học từ thông tin **không có sẵn tại thời điểm dự đoán**. Mô hình như vậy đạt điểm cao khi thử nghiệm nhưng không dùng được trong thực tế.

Thuộc tính `duration` là thời lượng cuộc gọi cuối. Ô dưới đây cho thấy `duration` liên quan rất mạnh đến nhãn. Câu hỏi then chốt: **ngân hàng có biết `duration` trước khi gọi không?**

`[MINH HỌA]` (dùng cho **Bảng 6**)

In [ ]:
theo_thoi_luong = train.groupby(pd.cut(train["duration"], [-1, 60, 120, 300, 600, 5000]), observed=True)["dong_y"].agg(["mean", "size"])
print("Số khách có duration = 0:", int((train["duration"] == 0).sum()), "| trong đó số khách đồng ý:", int(train.loc[train["duration"] == 0, "dong_y"].sum()))
theo_thoi_luong.round(3).rename_axis("duration (giây)").T

`[BÁO CÁO]` Trong 3–4 câu: vì sao `duration` dự đoán nhãn rất tốt; vì sao đồ án **loại** `duration` khỏi các thuộc tính đầu vào (biến `THUOC_TINH_SO` ở ô thiết lập không có `duration`). Tuần 2 sẽ đo xem nếu giữ `duration` thì điểm số "đẹp" lên bao nhiêu.

*Trả lời của sinh viên:* …

## 1.7 `[KIẾN THỨC]` Thước đo cho lớp yes
Với lớp `yes` là lớp cần phát hiện, gọi **TP** là số khách đồng ý được dự đoán đúng, **FN** là số khách đồng ý bị bỏ lỡ, **FP** là số khách bị gọi nhầm:

$$\text{Precision-yes}=\frac{TP}{TP+FP},\qquad \text{Recall-yes}=\frac{TP}{TP+FN},\qquad \text{F1-yes}=\frac{2\,TP}{2\,TP+FP+FN}.$$

- Precision-yes: trong các khách được chọn để gọi, tỉ lệ khách đồng ý, tức **tỉ lệ cuộc gọi thành công**.
- Recall-yes: trong các khách sẽ đồng ý, tỉ lệ khách **được chọn để gọi**.
- Hàm `chi_so` ở ô thiết lập gom các chỉ số chính thành một dòng để lập bảng; `classification_report` in đầy đủ cho từng lớp.

## 1.8 `[YÊU CẦU]` Mốc E0: luôn dự đoán lớp đa số
Trước khi xây dựng mô hình, cần một **mốc so sánh**: mô hình luôn dự đoán nhãn xuất hiện nhiều nhất trong **train**, tức `DummyClassifier(strategy="most_frequent")` của scikit-learn.
Yêu cầu sinh viên thực hiện: huấn luyện E0 trên train, dự đoán trên validation và tính các chỉ số.

In [ ]:
from sklearn.dummy import DummyClassifier
e0 = None          # YÊU CẦU: DummyClassifier(strategy="most_frequent"), huấn luyện bằng .fit(train, train["y"])
y_val_e0 = None    # YÊU CẦU: dự đoán trên val
kq_e0 = None       # YÊU CẦU: chi_so(val["y"], y_val_e0)

In [ ]:
# [TỰ KIỂM TRA]
assert kq_e0 is not None and len(y_val_e0) == len(val), "CHƯA ĐẠT: cần một dự đoán cho mỗi khách trong validation."
assert set(y_val_e0) == {train["y"].mode()[0]}, "CHƯA ĐẠT: E0 phải dự đoán lớp đa số của TRAIN."
print("ĐẠT: đã đánh giá mốc E0.", {k: round(float(v), 3) for k, v in kq_e0.items()})
ConfusionMatrixDisplay.from_predictions(val["y"], y_val_e0, labels=["yes", "no"]); plt.show()

`[CÂU HỎI SUY NGẪM]` (Câu 2) E0 có Accuracy khoảng 0,88 nhưng F1-yes bằng 0. Vì sao? Nếu chỉ báo cáo Accuracy, người đọc sẽ hiểu sai điều gì?

`[BÁO CÁO]` Ghi Accuracy, Recall-yes, F1-yes của E0 và trả lời câu hỏi trên trong 2–3 câu.

*Trả lời của sinh viên:* …

## Nộp cuối tuần 1
- [ ] Mọi ô `[TỰ KIỂM TRA]` in **ĐẠT**.
- [ ] Đã ghi: tỉ lệ yes của ba tập; ba nhận xét EDA (có một nhận xét về tuổi); đoạn giải thích về `duration`; kết quả và nhận xét về E0.
- [ ] **Viết nháp báo cáo:** Mục 1, Mục 2 (Bảng 1–6, Hình 1–2); dòng tuần 1 của Phụ lục A; Hình B.1 (Phụ lục B).
- [ ] Tải notebook đã chạy (*File → Download → Download .ipynb*), đặt tên `MSSV_tuan1.ipynb` và **lưu lại**: cả bốn notebook được nộp một lần cùng báo cáo vào cuối tuần 4 (không có đợt nộp giữa kỳ).

---
### Gợi ý trả lời các câu hỏi suy ngẫm
<details><summary>Mở phần này sau khi đã tự trả lời</summary>

**Câu 1.** Chưa nên. Tháng 3 chỉ có vài trăm khách được gọi, trong khi tháng 5 có hàng nghìn khách: chiến dịch tháng 5 gọi đại trà, các tháng ít cuộc gọi có thể là các đợt gọi chọn lọc (ví dụ gọi lại khách đã đồng ý trước đây). Dữ liệu được sắp xếp theo thời gian 2008–2010, nên tháng còn lẫn với năm và cách làm của từng chiến dịch. Tỉ lệ cao đi kèm với tháng 3 là một **tương quan**, chưa phải **nguyên nhân**.

**Câu 2.** Khoảng 88% khách trong validation là no. E0 luôn dự đoán no nên đúng khoảng 88% số khách, nhưng không chọn được khách đồng ý nào (Recall-yes = 0, do đó F1-yes = 0). Accuracy chỉ phản ánh tỉ lệ lớp, không phản ánh khả năng tìm khách tiềm năng — mục tiêu của bài toán.
</details>